# Liu2024 Source MAT - S-JEPA x Riemannian Hybrid

Combines the two methods at the **feature level**, then classifies with a shrinkage-LDA so the pipeline
stays data-efficient (no SGD head -> no collapse):

- **S-JEPA branch:** the *frozen pretrained* local encoder (channel-agnostic) -> per-channel feature maps
  -> mean-pooled over time tokens -> embedding vector.
- **Riemannian branch:** filter-bank spatial covariances -> tangent-space vectors (concatenated over bands).
  Uses a **faithful-strong** front-end (MI-marker aligned 0-4 s @ 500 Hz, **no average reference** so the
  covariance stays full rank, 8 broad bands) so S-JEPA is compared against the true honest ceiling, not a
  handicapped baseline.
- **Fusion:** standardize each branch on the train fold, concatenate, shrinkage-LDA.

Reports **riemannian / sjepa / fusion** on the same within-subject CV folds. Structure and conventions
follow the PreLocal notebook.

> Requires pyRiemann: `pip install pyriemann`

# 1. Setup

In [1]:
import os
import re
import sys
import json
import math
import hashlib
import random
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import torch

from scipy.io import loadmat

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import balanced_accuracy_score, accuracy_score, confusion_matrix

from braindecode.models import SignalJEPA, SignalJEPA_PreLocal

try:
    from pyriemann.estimation import Covariances
    from pyriemann.tangentspace import TangentSpace
except Exception as exc:
    raise ImportError("pyRiemann is required: pip install pyriemann") from exc

import mne

mne.set_log_level("WARNING")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Runtime Environment:")
print(f"  - Python:   {sys.version.split()[0]}")
print(f"  - Platform: {platform.platform()}")
print(f"  - Torch:    {torch.__version__}")


Runtime Environment:
  - Python:   3.11.15
  - Platform: macOS-26.5.1-arm64-arm-64bit
  - Torch:    2.10.0


# 2. Configuration

## 2.1 Liu2024 Channel Defaults

In [3]:
# Source files are trials x 33 channels x samples:
#   0..29 = EEG-like, index 17 = CPz source reference, 30..31 = EOG, 32 = marker.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

SOURCE_MARKER_CHANNEL_INDEX = 32

EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
                     if idx != SOURCE_REFERENCE_INDEX]


## 2.2 CONFIG

In [4]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-sjepa-riemannian-hybrid"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "sjepa_riemannian_hybrid",
    "config_note": "Frozen S-JEPA embedding + faithful-strong Riemannian tangent features -> shrinkage LDA.",

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "exclude_subjects": [],
    "source_unit": "microvolts",

    # ------------------------------------------------------------------
    # S-JEPA branch (frozen pretrained local encoder -> embedding)
    # The local encoder needs the 537-sample 128 Hz window it was pretrained with.
    # ------------------------------------------------------------------
    "sjepa": {
        "pretrained_mode": "from_pretrained",        # from_pretrained, random (control)
        "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
        "reference_mode": "average",
        "resample_sfreq": 128,
        "filter_low": 0.5,
        "filter_high": 40.0,
        "mi_window_start_s": 1.5,
        "target_window_samples": 537,
    },

    # ------------------------------------------------------------------
    # Riemannian branch (faithful-strong: matches the honest TWFB front-end)
    #   - MI-marker aligned, 0-4 s post-cue at the native 500 Hz (2000 samples)
    #   - NO average reference  -> covariance stays full rank
    #   - 8 broad bands from the Liu .m
    # Decoupled from the S-JEPA window on purpose, so each branch gets its best input.
    # ------------------------------------------------------------------
    "riemann": {
        "use_marker_alignment": True,
        "marker_channel_index0": 32,
        "mi_marker_value": 2,
        "fallback_onset_sample": 1000,               # 2.0 s @ 500 Hz if no marker found
        "window_len_samples": 2000,                  # 0-4 s @ 500 Hz
        "reference_mode": "none",                    # none, average
        "notch_freq": 50.0,
        "filter_bands": [[8, 12], [8, 20], [8, 30], [12, 20],
                         [15, 20], [15, 30], [20, 30], [8, 15]],
        "cov_estimator": "oas",                      # oas, scm
    },

    # ------------------------------------------------------------------
    # Classifier / evaluation
    # ------------------------------------------------------------------
    "lda_shrinkage": "auto",
    "evaluation_mode": "stratified_kfold",
    "cv_folds": 5,
    "collapse_threshold": 0.9,

    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "cv_random_state": 2026,
}


## 2.3 Constants and Derived Settings

In [5]:
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
TARGET_N_CLASSES = 2

N_CH = len(EEG_CHANNEL_NAMES)
SJEPA_SFREQ = float(CONFIG["sjepa"]["resample_sfreq"])
WINDOW_SAMPLES = int(CONFIG["sjepa"]["target_window_samples"])
RIEM_SFREQ = float(LIU_SOURCE_SFREQ)
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])

print("Effective hybrid settings:")
print(f"  Channels:                {N_CH}")
print(f"  S-JEPA window:           {WINDOW_SAMPLES} samples @ {SJEPA_SFREQ} Hz "
      f"(start {CONFIG['sjepa']['mi_window_start_s']} s)")
print(f"  Riemannian window:       {CONFIG['riemann']['window_len_samples']} samples @ {RIEM_SFREQ} Hz "
      f"(marker-aligned, ref={CONFIG['riemann']['reference_mode']})")
print(f"  Riemannian bands:        {len(CONFIG['riemann']['filter_bands'])}")
print(f"  Evaluation:              {CONFIG['evaluation_mode']} | {CONFIG['cv_folds']}-fold within-subject")
print(f"  S-JEPA pretrained_mode:  {CONFIG['sjepa']['pretrained_mode']}")


Effective hybrid settings:
  Channels:                29
  S-JEPA window:           537 samples @ 128.0 Hz (start 1.5 s)
  Riemannian window:       2000 samples @ 500.0 Hz (marker-aligned, ref=none)
  Riemannian bands:        8
  Evaluation:              stratified_kfold | 5-fold within-subject
  S-JEPA pretrained_mode:  from_pretrained


## 2.4 Artifact Creation and Logging Init

In [6]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")


Run ID:     20260612_1149_39ca8f85
Artifacts:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-sjepa-riemannian-hybrid/20260612_1149_39ca8f85


## 2.5 Reproducibility

In [7]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")


Using device: mps
Seed initialized: 2026


# 3. Load and Prepare Data

## 3.1 Data Loading Helpers

In [8]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    m = re.search(r"sub[-_ ]?(\d{1,2})", str(path), flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if not nums:
        raise ValueError(f"Could not infer subject id from {path}")
    return int(nums[-1])

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def _normalize_rawdata_shape(rawdata, labels=None):
    rawdata = np.asarray(rawdata)
    if rawdata.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got {rawdata.shape}")
    trial_axes = []
    if labels is not None:
        n = int(np.asarray(labels).size)
        trial_axes = [ax for ax, s in enumerate(rawdata.shape) if s == n]
    if not trial_axes:
        trial_axes = [ax for ax, s in enumerate(rawdata.shape) if s in (39, 40)]
    if trial_axes and trial_axes[0] != 0:
        rawdata = np.moveaxis(rawdata, trial_axes[0], 0)
    time_axis = int(np.argmax(rawdata.shape[1:]) + 1)
    if time_axis != 2:
        rawdata = np.moveaxis(rawdata, time_axis, 2)
    return rawdata

def load_subject_mat(path):
    """Return rawdata as trials x 33 channels x samples (keeps marker channel) and integer labels."""
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    arrays = [(n, np.asarray(v)) for n, v in _walk_mat_object(mat)
              if isinstance(v, np.ndarray) and v.dtype != object]
    raw_candidates = [a for _, a in arrays if a.ndim == 3]
    label_candidates = [a for _, a in arrays if a.ndim in (1, 2) and np.asarray(a).size in (39, 40)]
    if not raw_candidates or not label_candidates:
        raise KeyError(f"Could not locate rawdata/labels in {path}")
    rawdata = max(raw_candidates, key=lambda a: max(a.shape))
    labels = np.asarray(label_candidates[0]).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(rawdata, labels).astype(np.float64)
    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label/trial mismatch in {path}: {labels.shape} vs {rawdata.shape}")
    return rawdata, labels

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")


## 3.2 Two Preprocessing Paths

`preprocess_sjepa` reproduces the PreLocal front-end (average reference, resample 128 Hz, FIR 0.5-40 Hz,
fixed 537-sample window) because the frozen encoder expects exactly that. `preprocess_riemann` produces
the faithful-strong covariance front-end (MI-marker aligned 0-4 s at 500 Hz, no average reference). The
two are intentionally independent.

In [9]:
def make_liu_info(sfreq):
    info = mne.create_info(EEG_CHANNEL_NAMES, float(sfreq), ["eeg"] * len(EEG_CHANNEL_NAMES))
    info.set_montage(mne.channels.make_standard_montage("standard_1020"), match_case=False, on_missing="ignore")
    return info

def _source_to_volts(x):
    return np.asarray(x, np.float64) * (1e-6 if str(CONFIG["source_unit"]).startswith("micro") else 1.0)

def _volts_to_microvolts(x):
    return np.asarray(x, np.float64) * 1e6

def preprocess_sjepa(rawdata, labels):
    X = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)          # trials x 29 x 4000 @ 500 Hz
    n = X.shape[0]
    cont = _source_to_volts(X).transpose(1, 0, 2).reshape(N_CH, -1)
    raw = mne.io.RawArray(cont, make_liu_info(LIU_SOURCE_SFREQ), verbose=False)
    if CONFIG["sjepa"]["reference_mode"] == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
    raw.resample(SJEPA_SFREQ, verbose=False)
    raw.filter(CONFIG["sjepa"]["filter_low"], CONFIG["sjepa"]["filter_high"],
               method="fir", phase="zero", fir_design="firwin", verbose=False)
    data = _volts_to_microvolts(raw.get_data())
    per = data.shape[1] // n
    data = data[:, :n * per]
    Xrs = data.reshape(N_CH, n, per).transpose(1, 0, 2)               # trials x 29 x per @ 128 Hz
    start = int(round(CONFIG["sjepa"]["mi_window_start_s"] * SJEPA_SFREQ))
    stop = start + WINDOW_SAMPLES
    if stop > Xrs.shape[-1]:
        raise ValueError(f"S-JEPA window [{start}:{stop}] exceeds {Xrs.shape[-1]} samples")
    return Xrs[:, :, start:stop].astype(np.float32), labels_to_zero_based(labels)

def _find_mi_onset(marker_row):
    hits = np.where(np.round(marker_row).astype(int) == CONFIG["riemann"]["mi_marker_value"])[0]
    return int(hits[0]) if hits.size else int(CONFIG["riemann"]["fallback_onset_sample"])

def preprocess_riemann(rawdata, labels):
    cfg = CONFIG["riemann"]
    X = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)          # trials x 29 x 4000 @ 500 Hz
    n = X.shape[0]
    W = int(cfg["window_len_samples"])
    out = np.empty((n, N_CH, W), dtype=np.float64)
    for t in range(n):
        if cfg["use_marker_alignment"]:
            onset = _find_mi_onset(rawdata[t, cfg["marker_channel_index0"], :])
        else:
            onset = int(cfg["fallback_onset_sample"])
        s0, s1 = onset, onset + W
        if s1 > X.shape[2]:
            s1 = X.shape[2]; s0 = s1 - W
        out[t] = X[t, :, s0:s1]
    if cfg["reference_mode"] == "average":
        out = out - out.mean(axis=1, keepdims=True)
    if cfg.get("notch_freq"):
        out = mne.filter.notch_filter(out, RIEM_SFREQ, freqs=[float(cfg["notch_freq"])], verbose=False)
    return out.astype(np.float64), labels_to_zero_based(labels)


## 3.3 Load and Preprocess Every Subject

In [10]:
MAT_FILES = find_source_mat_files(SOURCE_EXTRACT_DIR)
if not MAT_FILES:
    raise FileNotFoundError(f"No .mat files under {SOURCE_EXTRACT_DIR}")

use = None if CONFIG["subjects_to_use"] is None else {int(s) for s in CONFIG["subjects_to_use"]}
excl = {int(s) for s in CONFIG["exclude_subjects"]}

EEG_INFO = make_liu_info(SJEPA_SFREQ)
CHS_INFO = EEG_INFO["chs"]
CH_NAMES = list(EEG_CHANNEL_NAMES)

SUBJECT_DATA = {}
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if (use is not None and sid not in use) or sid in excl:
        continue
    rawdata, labels = load_subject_mat(p)
    X_sjepa, y = preprocess_sjepa(rawdata, labels)
    X_riem, y2 = preprocess_riemann(rawdata, labels)
    assert np.array_equal(y, y2), "label mismatch between preprocessing paths"
    SUBJECT_DATA[sid] = {"X_sjepa": X_sjepa, "X_riem": X_riem, "y": y.astype(int)}

ALL_SUBJECTS = sorted(SUBJECT_DATA)
example = SUBJECT_DATA[ALL_SUBJECTS[0]]
print(f"Loaded {len(ALL_SUBJECTS)} subjects")
print(f"  X_sjepa: {example['X_sjepa'].shape}  (trials x ch x time @ {SJEPA_SFREQ} Hz)")
print(f"  X_riem:  {example['X_riem'].shape}  (trials x ch x time @ {RIEM_SFREQ} Hz)")
print(f"  classes: {np.bincount(example['y']).tolist()}")


/var/folders/7d/njk_0cn503z09dk98r0csptr0000gn/T/ipykernel_6358/3575023819.py:54: RuntimeWarning: filter_length (3301) is longer than the signal (2000), distortion is likely. Reduce filter length or filter a longer signal.
  out = mne.filter.notch_filter(out, RIEM_SFREQ, freqs=[float(cfg["notch_freq"])], verbose=False)
/var/folders/7d/njk_0cn503z09dk98r0csptr0000gn/T/ipykernel_6358/3575023819.py:54: RuntimeWarning: filter_length (3301) is longer than the signal (2000), distortion is likely. Reduce filter length or filter a longer signal.
  out = mne.filter.notch_filter(out, RIEM_SFREQ, freqs=[float(cfg["notch_freq"])], verbose=False)
/var/folders/7d/njk_0cn503z09dk98r0csptr0000gn/T/ipykernel_6358/3575023819.py:54: RuntimeWarning: filter_length (3301) is longer than the signal (2000), distortion is likely. Reduce filter length or filter a longer signal.
  out = mne.filter.notch_filter(out, RIEM_SFREQ, freqs=[float(cfg["notch_freq"])], verbose=False)
/var/folders/7d/njk_0cn503z09dk98r0cs

Loaded 50 subjects
  X_sjepa: (40, 29, 537)  (trials x ch x time @ 128.0 Hz)
  X_riem:  (40, 29, 2000)  (trials x ch x time @ 500.0 Hz)
  classes: [20, 20]


/var/folders/7d/njk_0cn503z09dk98r0csptr0000gn/T/ipykernel_6358/3575023819.py:54: RuntimeWarning: filter_length (3301) is longer than the signal (2000), distortion is likely. Reduce filter length or filter a longer signal.
  out = mne.filter.notch_filter(out, RIEM_SFREQ, freqs=[float(cfg["notch_freq"])], verbose=False)


# 4. Feature Extractors

## 4.1 S-JEPA Frozen Embeddings

Fix for the `feature_encoder` shape error: in `SignalJEPA_PreLocal` the `feature_encoder` sits after the
29->4 `spatial_conv`, so it rejects raw 29-channel input. We instead use the **base `SignalJEPA`**, whose
`feature_encoder` is channel-agnostic and accepts `[B, 29, T]`, and copy the pretrained local-encoder
weights into it. No training, no random spatial layer, so this is purely the pretrained representation.

In [11]:
def _construct_base_sjepa():
    last = None
    for kw in (
        dict(n_times=WINDOW_SAMPLES, chs_info=CHS_INFO, sfreq=SJEPA_SFREQ),
        dict(n_chans=N_CH, n_times=WINDOW_SAMPLES, sfreq=SJEPA_SFREQ),
        dict(n_chans=N_CH, n_times=WINDOW_SAMPLES, chs_info=CHS_INFO),
        dict(n_times=WINDOW_SAMPLES, chs_info=CHS_INFO),
    ):
        try:
            return SignalJEPA(**kw)
        except Exception as exc:  # noqa: BLE001
            last = exc
    raise RuntimeError(f"Could not construct base SignalJEPA: {last}")

def build_sjepa_encoder():
    enc = _construct_base_sjepa()
    if CONFIG["sjepa"]["pretrained_mode"] == "from_pretrained":
        pre = SignalJEPA_PreLocal.from_pretrained(
            CONFIG["sjepa"]["pretrained_repo_id"],
            n_chans=N_CH, chs_info=CHS_INFO, n_times=WINDOW_SAMPLES,
            n_outputs=TARGET_N_CLASSES, strict=False,
        )
        missing, unexpected = enc.feature_encoder.load_state_dict(pre.feature_encoder.state_dict(), strict=False)
        loaded = sum(p.numel() for p in enc.feature_encoder.parameters())
        print(f"Copied pretrained feature_encoder: {loaded:,} params "
              f"(missing={len(missing)}, unexpected={len(unexpected)})")
        if loaded == 0:
            raise RuntimeError("feature_encoder has 0 params after copy - check braindecode version.")
    else:
        print("Using RANDOM-initialised feature_encoder (control).")
    return enc.to(DEVICE).eval()

ENCODER = build_sjepa_encoder()

# probe token geometry
with torch.no_grad():
    _probe = ENCODER.feature_encoder(torch.zeros(1, N_CH, WINDOW_SAMPLES, device=DEVICE))
L_TOK, D_MODEL = int(_probe.shape[1]), int(_probe.shape[2])
assert L_TOK % N_CH == 0, f"token count {L_TOK} not divisible by {N_CH} channels"
N_TOK_PER_CH = L_TOK // N_CH
EMBED_DIM = N_CH * D_MODEL
print(f"feature_encoder output: {L_TOK} tokens x {D_MODEL} dims -> {N_TOK_PER_CH} tokens/channel "
      f"-> embedding dim {EMBED_DIM}")

@torch.no_grad()
def sjepa_embeddings(X):
    """X: trials x C x T (float32) -> trials x (C*d), mean-pooled over time tokens per channel."""
    xb = torch.as_tensor(np.asarray(X, np.float32), device=DEVICE)
    feat = ENCODER.feature_encoder(xb)                       # [N, C*t, d] (channel-major)
    n = feat.shape[0]
    feat = feat.reshape(n, N_CH, N_TOK_PER_CH, D_MODEL).mean(dim=2)   # [N, C, d]
    return feat.reshape(n, -1).cpu().numpy().astype(np.float64)


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/braindecode/models/signal_jepa.py:1169: RuntimeWarning: divide by zero encountered in scalar divide
  xx = (x - x_min) / (x_max - x_min)


Copied pretrained feature_encoder: 13,840 params (missing=0, unexpected=0)
feature_encoder output: 116 tokens x 64 dims -> 4 tokens/channel -> embedding dim 1856


## 4.2 Riemannian Tangent-Space Features

Per band: band-pass -> spatial covariance -> tangent-space projection. The `TangentSpace` reference is fit
on the training fold only; train and test are projected through the fitted reference, then concatenated
across bands.

In [12]:
def _bandpass(X, lo, hi):
    return mne.filter.filter_data(np.asarray(X, np.float64), RIEM_SFREQ, lo, hi,
                                  method="fir", phase="zero", fir_design="firwin", verbose=False)

def fit_riemann_features(X_train):
    fitted = []
    for (lo, hi) in CONFIG["riemann"]["filter_bands"]:
        Xb = _bandpass(X_train, lo, hi)
        cov = Covariances(estimator=CONFIG["riemann"]["cov_estimator"]).transform(Xb)
        ts = TangentSpace().fit(cov)
        fitted.append(((lo, hi), ts))
    return fitted

def transform_riemann_features(X, fitted):
    feats = []
    for (lo, hi), ts in fitted:
        Xb = _bandpass(X, lo, hi)
        cov = Covariances(estimator=CONFIG["riemann"]["cov_estimator"]).transform(Xb)
        feats.append(ts.transform(cov))
    return np.concatenate(feats, axis=1)


# 5. Evaluation

## 5.1 Per-fold Pipelines

Each branch is standardized on the train fold and classified with a shrinkage-LDA (data-efficient, cannot
collapse). Fusion concatenates the two standardized feature vectors.

In [13]:
def make_lda():
    return LinearDiscriminantAnalysis(solver="lsqr", shrinkage=CONFIG["lda_shrinkage"])

def _collapse_ratio(y_pred):
    hist = np.bincount(np.asarray(y_pred, int), minlength=TARGET_N_CLASSES)
    return float(hist.max() / hist.sum()) if hist.sum() else 0.0


## 5.2 Subject Cross-Validation Runner

In [14]:
def run_subject(sid):
    data = SUBJECT_DATA[sid]
    X_sjepa, X_riem, y = data["X_sjepa"], data["X_riem"], data["y"]
    skf = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=CONFIG["cv_random_state"])

    preds = {"riemannian": [], "sjepa": [], "fusion": []}
    truth = []
    for tr, te in skf.split(X_sjepa, y):
        truth.extend(y[te].tolist())

        # Riemannian branch
        fitted = fit_riemann_features(X_riem[tr])
        R_tr = transform_riemann_features(X_riem[tr], fitted)
        R_te = transform_riemann_features(X_riem[te], fitted)
        sR = StandardScaler().fit(R_tr)
        R_tr, R_te = sR.transform(R_tr), sR.transform(R_te)

        # S-JEPA branch
        E_tr = sjepa_embeddings(X_sjepa[tr])
        E_te = sjepa_embeddings(X_sjepa[te])
        sE = StandardScaler().fit(E_tr)
        E_tr, E_te = sE.transform(E_tr), sE.transform(E_te)

        preds["riemannian"].extend(make_lda().fit(R_tr, y[tr]).predict(R_te).tolist())
        preds["sjepa"].extend(make_lda().fit(E_tr, y[tr]).predict(E_te).tolist())

        F_tr = np.concatenate([R_tr, E_tr], axis=1)
        F_te = np.concatenate([R_te, E_te], axis=1)
        preds["fusion"].extend(make_lda().fit(F_tr, y[tr]).predict(F_te).tolist())

    y_true = np.asarray(truth, int)
    out = {"subject": int(sid)}
    for key in preds:
        y_pred = np.asarray(preds[key], int)
        out[f"{key}_balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
        out[f"{key}_accuracy"] = float(accuracy_score(y_true, y_pred))
        out[f"{key}_collapse_ratio"] = _collapse_ratio(y_pred)
        out[f"{key}_pred"] = y_pred.tolist()
    out["y_true"] = y_true.tolist()
    return out


## 5.3 Run All Subjects

In [15]:
import time
RESULTS = []
t0 = time.time()
for i, sid in enumerate(ALL_SUBJECTS, 1):
    r = run_subject(sid)
    RESULTS.append(r)
    print(f"[{i:2d}/{len(ALL_SUBJECTS)}] subj {sid:2d}  "
          f"riem={r['riemannian_balanced_accuracy']*100:5.1f}  "
          f"sjepa={r['sjepa_balanced_accuracy']*100:5.1f}  "
          f"fusion={r['fusion_balanced_accuracy']*100:5.1f}  "
          f"({(time.time()-t0)/60:.1f} min)")

with open(ARTIFACT_DIR / "hybrid_results.json", "w") as f:
    json.dump(RESULTS, f, indent=2)


[ 1/50] subj  1  riem= 57.5  sjepa= 57.5  fusion= 70.0  (4.6 min)
[ 2/50] subj  2  riem= 42.5  sjepa= 45.0  fusion= 32.5  (9.2 min)
[ 3/50] subj  3  riem= 45.0  sjepa= 65.0  fusion= 52.5  (13.8 min)
[ 4/50] subj  4  riem= 47.5  sjepa= 55.0  fusion= 42.5  (18.4 min)
[ 5/50] subj  5  riem= 55.0  sjepa= 57.5  fusion= 55.0  (22.9 min)
[ 6/50] subj  6  riem= 42.5  sjepa= 37.5  fusion= 32.5  (27.5 min)
[ 7/50] subj  7  riem= 75.0  sjepa= 57.5  fusion= 77.5  (32.1 min)
[ 8/50] subj  8  riem= 35.0  sjepa= 60.0  fusion= 42.5  (36.6 min)
[ 9/50] subj  9  riem= 50.0  sjepa= 52.5  fusion= 47.5  (40.9 min)
[10/50] subj 10  riem= 57.5  sjepa= 75.0  fusion= 65.0  (45.2 min)
[11/50] subj 11  riem= 70.0  sjepa= 65.0  fusion= 67.5  (49.5 min)
[12/50] subj 12  riem= 45.0  sjepa= 52.5  fusion= 50.0  (53.8 min)
[13/50] subj 13  riem= 52.5  sjepa= 62.5  fusion= 60.0  (58.1 min)
[14/50] subj 14  riem= 42.5  sjepa= 47.5  fusion= 40.0  (62.4 min)
[15/50] subj 15  riem= 62.5  sjepa= 45.0  fusion= 55.0  (66.6 mi

# 6. Results

## 6.1 Aggregate and Side-by-side Comparison

In [16]:
def _global_balanced_accuracy(key):
    y_true = np.concatenate([r["y_true"] for r in RESULTS])
    y_pred = np.concatenate([r[f"{key}_pred"] for r in RESULTS])
    return balanced_accuracy_score(y_true, y_pred)

rows = []
for key in ["riemannian", "sjepa", "fusion"]:
    arr = np.array([r[f"{key}_balanced_accuracy"] for r in RESULTS])
    coll = np.array([r[f"{key}_collapse_ratio"] >= CONFIG["collapse_threshold"] for r in RESULTS])
    rows.append({
        "pipeline": key,
        "mean_per_subject_BA_%": round(arr.mean() * 100, 2),
        "SD_%": round(arr.std() * 100, 2),
        "global_pooled_BA_%": round(_global_balanced_accuracy(key) * 100, 2),
        "collapsed_subjects": int(coll.sum()),
    })
summary = pd.DataFrame(rows)

print("=" * 70)
print(f"S-JEPA x Riemannian hybrid | {len(ALL_SUBJECTS)} subjects | "
      f"{CONFIG['cv_folds']}-fold within-subject | sjepa={CONFIG['sjepa']['pretrained_mode']}")
print(summary.to_string(index=False))
print("-" * 70)
print("Reference (honest): CSP+LDA ~55.6 | FBCSP+SVM ~57.6 | TWFB honest ~53-55")
print("Reference (leaky):  Liu reported TWFB+DGFMDM 72.21 (test-set band selection)")
print("=" * 70)

summary.to_csv(ARTIFACT_DIR / "hybrid_summary.csv", index=False)

per_subject = pd.DataFrame([{
    "subject": r["subject"],
    "riemannian_%": round(r["riemannian_balanced_accuracy"] * 100, 1),
    "sjepa_%": round(r["sjepa_balanced_accuracy"] * 100, 1),
    "fusion_%": round(r["fusion_balanced_accuracy"] * 100, 1),
} for r in RESULTS]).sort_values("subject").reset_index(drop=True)
per_subject.to_csv(ARTIFACT_DIR / "hybrid_per_subject.csv", index=False)
summary


S-JEPA x Riemannian hybrid | 50 subjects | 5-fold within-subject | sjepa=from_pretrained
  pipeline  mean_per_subject_BA_%  SD_%  global_pooled_BA_%  collapsed_subjects
riemannian                  53.95 12.81               53.95                   0
     sjepa                  55.05  9.06               55.05                   0
    fusion                  55.90 12.04               55.90                   0
----------------------------------------------------------------------
Reference (honest): CSP+LDA ~55.6 | FBCSP+SVM ~57.6 | TWFB honest ~53-55
Reference (leaky):  Liu reported TWFB+DGFMDM 72.21 (test-set band selection)


,pipeline,mean_per_subject_BA_%,SD_%,global_pooled_BA_%,collapsed_subjects
0,riemannian,53.95,12.81,53.95,0
1,sjepa,55.05,9.06,55.05,0
2,fusion,55.90,12.04,55.90,0


## 6.2 Notes

- **fusion vs riemannian**: does adding S-JEPA features help on top of the (now fair) Riemannian baseline?
- **sjepa vs riemannian**: how far the frozen pretrained features get with a data-efficient head - and
  whether that clears the ~57% collapse-prone SGD-head runs.
- **pretrained vs random**: set `CONFIG["sjepa"]["pretrained_mode"] = "random"` to confirm the S-JEPA
  contribution is real rather than architectural.
- The Riemannian branch here is honest (inner CV is implicit in the shrinkage-LDA fit; no test-set band
  selection). For the standalone per-subject TW+FB-selected FgMDM, use the TWFB reproduction notebook.